In [ ]:
# Feature Engineering (Feature Selection + Dimension Reduction)
# Source:IT24102298_Dimension_reduction.ipynb
# Assigned to: IT24102298
# Description: Selects top 8 features using SelectKBest and reduces dimensionality to 5 principal components using PCA.

In [ ]:
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

start_time = time.time()
print("Starting Feature Engineering (Feature Selection + Dimension Reduction) at", time.strftime("%H:%M:%S", time.localtime()))

# Load dataset 
df = pd.read_csv('../results/outputs/scaled_data.csv')
print("Dataset loaded. Shape:", df.shape)

# Drop identifier columns (non-numeric, e.g., Student_ID, names, email) to avoid conversion errors
df = df.drop(['Student_ID', 'First_Name', 'Last_Name', 'Email'], axis=1, errors='ignore')
print("Dropped identifiers. Updated shape:", df.shape)

# Feature Selection: SelectKBest
X = df.drop(['Total_Score', 'Grade'], axis=1)
y = df['Total_Score']
selector = SelectKBest(score_func=f_regression, k=8)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
print("Selected features:", selected_features)

# Create DataFrame with selected features
df_selected = pd.DataFrame(X_selected, columns=selected_features, index=df.index)
df_selected['Total_Score'] = y
print("Feature selection completed. Shape:", df_selected.shape)

# Dimension Reduction: PCA
numeric_cols = df_selected.select_dtypes(include=['float64', 'int64']).columns
X = df_selected[numeric_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Data scaled for PCA. Shape:", X_scaled.shape)
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)
explained_variance = pca.explained_variance_ratio_.sum()
print(f"PCA explained variance: {explained_variance:.2f}")
X_reduced = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(5)], index=df_selected.index)
print("Dimension reduction completed.")

# Merge reduced components back with Total_Score
df_pca = pd.concat([df_selected[['Total_Score']], X_reduced], axis=1)
print("Merged PCA components with Total_Score. New shape:", df_pca.shape)

# EDA Visualization: Scatter Plot of PC1 vs PC2
print("\nEDA - Scatter Plot of Principal Components")
df_pca['At_Risk'] = df_pca['Total_Score'].apply(lambda s: 1 if s < 60 else 0)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='At_Risk', palette='husl', s=100)
plt.title('IT24102298: Scatter Plot of PC1 vs PC2 by At-Risk Status')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.savefig('../results/eda_visualizations/eda_IT24102298_scatterplot.png')
plt.close()
print("Saved scatter plot. Interpretation: Shows relationship between PC1 and PC2, colored by at-risk status to identify potential clusters.")

# Final Step: Save the final preprocessed data
df_pca.to_csv('../results/outputs/final_preprocessed_data.csv', index=False)
print("Final preprocessed data saved: preprocessed_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Feature Engineering Task Complete.")

Starting Feature Engineering (Feature Selection + Dimension Reduction) at 15:02:16
Dataset loaded. Shape: (4696, 30)
Dropped identifiers. Updated shape: (4696, 26)
Selected features: ['Midterm_Score', 'Final_Score', 'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score', 'Projects_Score', 'Average_Score', 'Study_Efficiency']
Feature selection completed. Shape: (4696, 9)
Data scaled for PCA. Shape: (4696, 9)
PCA explained variance: 0.79
Dimension reduction completed.
Merged PCA components with Total_Score. New shape: (4696, 6)

EDA - Scatter Plot of Principal Components
Saved scatter plot. Interpretation: Shows relationship between PC1 and PC2, colored by at-risk status to identify potential clusters.
Final preprocessed data saved: preprocessed_data.csv

Runtime: 1.62 seconds
Feature Engineering Task Complete.
